<a href="https://colab.research.google.com/github/samarulrajt/colab/blob/main/simple_chat_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate toolz

In [ ]:
import torch
from transformers import pipeline

print("அதிவேக மாடல் லோடு ஆகிறது...")

# 7B மாடலுக்குப் பதிலாக 3B மாடலைப் பயன்படுத்துகிறோம்.
# இது 100% இலவச GPU மெமரிக்குள் அடங்குவதால் புல்லட் வேகத்தில் இயங்கும்.
fast_pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    dtype=torch.bfloat16,
    device_map="auto"
)

print("மாடல் லோடு ஆகிவிட்டது! சாட் செய்யத் தயாராக உள்ளது.\n")

messages = [{"role": "system", "content": "You are a helpful AI assistant."}]

while True:
    user_input = input("நீங்கள்: ")
    if user_input.lower() in ['exit', 'quit']: break
    if not user_input.strip(): continue

    messages.append({"role": "user", "content": user_input})
    print("AI: ", end="", flush=True)

    # max_length=None கொடுத்துள்ளதால் வார்னிங் வராது, வேகமாகவும் பதில் வரும்
    outputs = fast_pipe(
        messages,
        max_new_tokens=256,
        temperature=0.7,
        max_length=None
    )

    assistant_response = outputs[0]["generated_text"][-1]["content"]
    print(assistant_response)
    print("\n" + "-"*50 + "\n")

    messages.append({"role": "assistant", "content": assistant_response})


In [ ]:
# 1. தேவையான டூல்களை நிறுவுதல்
!pip install -q fastapi uvicorn pydantic pyngrok

import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok

# கொலாபில் அசிங்க் இயங்க அனுமதிப்பது
nest_asyncio.apply()

app = FastAPI()

class ChatRequest(BaseModel):
    prompt: str

@app.post("/chat")
def chat_endpoint(request: ChatRequest):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": request.prompt}
    ]
    # நாம் ஏற்கனவே லோடு செய்த fast_pipe-ஐப் பயன்படுத்துகிறது
    outputs = fast_pipe(messages, max_new_tokens=256, max_length=None)
    return {"response": outputs[0]["generated_text"][-1]["content"]}

# 2. லோக்கல் கம்ப்யூட்டருடன் இணைக்க இலவச பொது லிங்க் (Public URL) உருவாக்குதல்
# (குறிப்பு: ngrok.com-ல் இலவச அக்கவுண்ட் தொடங்கி உங்கள் டோக்கனை இங்கு போடலாம், அல்லது நேரடியாக ரன் செய்யலாம்)
# உங்கள் ngrok authentication token ஐ இங்கு இட்டுக்கொள்ளவும்:
ngrok.set_auth_token("3JAmrjvk7GDN0M2CIi61yly8gyN_LRdhq2JPaCnwoMer2jQ6")
public_url = ngrok.connect(8000)
print(f"\n🔗 உங்கள் லோக்கல் கணினியுடன் இணைப்பதற்கான லிங்க்:\n{public_url.public_url}\n")

# சர்வரை ஆன் செய்தல்
import uvicorn
from uvicorn.config import Config
import asyncio

# Removed loop_factory=None as Config does not accept it.
config = Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

# Run the uvicorn server's serve() coroutine within the current event loop
# using asyncio.run(), which is patched by nest_asyncio.
asyncio.run(server.serve())

In [ ]:
import pyngrok

pyngrok.ngrok.kill()
print("All ngrok tunnels killed.")